In [ ]:
# Install and remove torchao
!pip uninstall -y torchao
!pip install -q peft accelerate

In [ ]:
# Imports and label encoding
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model
from datasets import Dataset
import warnings
warnings.filterwarnings("ignore")

SEED = 42
set_seed(SEED)

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

LETTER2ID = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train['label'] = train['answer'].map(LETTER2ID)

MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 128
NUM_OPTIONS = 5
OPTION_LETTERS = ['A', 'B', 'C', 'D', 'E']

In [ ]:
# Q1 encoded label at index 150
q1 = int(train.iloc[150]['label'])
print(f"Q1: {q1}")

In [ ]:
# Q2 row 0 option B length
row0 = train.iloc[0]
formatted_B_row0 = str(row0['prompt']) + " [SEP] " + str(row0['B'])
q2 = len(formatted_B_row0)
print(f"Q2: {q2}")

In [ ]:
# Load tokenizer and helpers
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def format_row_choices(row):
    p = str(row['prompt'])
    return [p + " [SEP] " + str(row[L]) for L in OPTION_LETTERS]

def tokenize_rows_for_mc(rows_df, max_length=MAX_LEN):
    all_texts = []
    for _, row in rows_df.iterrows():
        all_texts.extend(format_row_choices(row))
    enc = tokenizer(
        all_texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    )
    N = len(rows_df)
    return {
        'input_ids': enc['input_ids'].view(N, NUM_OPTIONS, max_length),
        'attention_mask': enc['attention_mask'].view(N, NUM_OPTIONS, max_length),
    }

In [ ]:
# Q3 single row tokenization shape
row0_enc = tokenize_rows_for_mc(train.iloc[[0]], max_length=MAX_LEN)
q3 = int(row0_enc['input_ids'].shape[1])
print(f"row 0 input_ids shape: {tuple(row0_enc['input_ids'].shape)}")
print(f"Q3: {q3}")

In [ ]:
# Q4 batch tokenization token count
batch16_enc = tokenize_rows_for_mc(train.iloc[:16], max_length=MAX_LEN)
q4 = int(batch16_enc['input_ids'].numel())
print(f"batch16 input_ids shape: {tuple(batch16_enc['input_ids'].shape)}")
print(f"Q4: {q4}")

In [ ]:
# Q5 model logits shape
device = 'cuda' if torch.cuda.is_available() else 'cpu'
mc_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(device)
mc_model.eval()

with torch.no_grad():
    out_row0 = mc_model(
        input_ids=row0_enc['input_ids'].to(device),
        attention_mask=row0_enc['attention_mask'].to(device),
    )
q5 = int(out_row0.logits.shape[1])
print(f"row 0 logits shape: {tuple(out_row0.logits.shape)}")
print(f"Q5: {q5}")

In [ ]:
# Q6 supervised loss dimensions
label_row0 = torch.tensor([int(train.iloc[0]['label'])], dtype=torch.long, device=device)
with torch.no_grad():
    out_row0_labeled = mc_model(
        input_ids=row0_enc['input_ids'].to(device),
        attention_mask=row0_enc['attention_mask'].to(device),
        labels=label_row0,
    )
loss = out_row0_labeled.loss
q6 = int(loss.dim())
print(f"loss: {loss.item()}, shape: {tuple(loss.shape)}, ndim: {loss.dim()}")
print(f"Q6: {q6}")

In [ ]:
# Q7 LoRA trainable parameter count
set_seed(SEED)
base_for_lora = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['query', 'value'],
    lora_dropout=0.1,
    bias='none',
    task_type=TaskType.SEQ_CLS,
)
peft_model_q7 = get_peft_model(base_for_lora, lora_cfg)
q7 = sum(p.numel() for p in peft_model_q7.parameters() if p.requires_grad)
print(f"Q7: {q7}")
peft_model_q7.print_trainable_parameters()

In [ ]:
# Q8 dataset first item choices
first100 = train.iloc[:100].reset_index(drop=True)
enc100 = tokenize_rows_for_mc(first100, max_length=MAX_LEN)

hf_records = [
    {
        'input_ids': enc100['input_ids'][i].tolist(),
        'attention_mask': enc100['attention_mask'][i].tolist(),
        'labels': int(first100.loc[i, 'label']),
    }
    for i in range(len(first100))
]
hf_dataset_100 = Dataset.from_list(hf_records)

first_item_input_ids = torch.tensor(hf_dataset_100[0]['input_ids'])
q8 = int(first_item_input_ids.shape[0])
print(f"first item input_ids shape: {tuple(first_item_input_ids.shape)}")
print(f"Q8: {q8}")

In [ ]:
# Q9 tiny LoRA fine-tune global step
MAX_LEN_Q9 = 64

def tokenize_rows_for_mc_len(rows_df, max_length):
    all_texts = []
    for _, row in rows_df.iterrows():
        all_texts.extend(format_row_choices(row))
    enc = tokenizer(
        all_texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    )
    N = len(rows_df)
    return (
        enc['input_ids'].view(N, NUM_OPTIONS, max_length),
        enc['attention_mask'].view(N, NUM_OPTIONS, max_length),
    )

set_seed(SEED)
first32 = train.iloc[:32].reset_index(drop=True)
ids32, mask32 = tokenize_rows_for_mc_len(first32, max_length=MAX_LEN_Q9)

records32 = [
    {
        'input_ids': ids32[i].tolist(),
        'attention_mask': mask32[i].tolist(),
        'labels': int(first32.loc[i, 'label']),
    }
    for i in range(len(first32))
]
train_ds32 = Dataset.from_list(records32)

set_seed(SEED)
base_q9 = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
peft_model_q9 = get_peft_model(base_q9, lora_cfg)

training_args = TrainingArguments(
    output_dir='./ms4_tiny_lora',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy='no',
    report_to='none',
    seed=SEED,
    learning_rate=5e-5,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=peft_model_q9,
    args=training_args,
    train_dataset=train_ds32,
)
train_result = trainer.train()
q9 = int(train_result.global_step)
print(f"Q9: {q9}")

In [ ]:
# Q10 softmax probability of option E
peft_model_q9.eval()
row0_ids_q10, row0_mask_q10 = tokenize_rows_for_mc_len(train.iloc[[0]], max_length=MAX_LEN_Q9)
peft_device = next(peft_model_q9.parameters()).device
with torch.no_grad():
    logits_row0 = peft_model_q9(
        input_ids=row0_ids_q10.to(peft_device),
        attention_mask=row0_mask_q10.to(peft_device),
    ).logits
probs_row0 = F.softmax(logits_row0, dim=-1).squeeze(0).cpu().numpy()
q10 = round(float(probs_row0[4]), 4)
print(f"row 0 probs (A, B, C, D, E): {probs_row0}")
print(f"Q10: {q10}")